In [3]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-11-09 10:16:25--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-11-09 10:16:25 (33.6 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [4]:
with open('input.txt','r',encoding='utf-8') as f:
  text = f.read()
text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [5]:
chars = sorted(list(set(text)))
print(chars)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [6]:
vocab_size = len(chars)
vocab_size

65

In [7]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [8]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100]) # the 100 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [9]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [10]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [12]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")
    print("----")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
----
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [4

In [13]:
print(xb) #(4,8)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [14]:
import torch
import torch.nn as nn

In [15]:
vocab_size = len(chars)
n_embedd = 256
block_size = 8
batch_size = 4

In [16]:
class Masked_self_attention(nn.Module):
  def __init__(self,n_embedd,head_size):
    super().__init__()
    self.key = nn.Linear(n_embedd,head_size,bias=False)
    self.querry = nn.Linear(n_embedd,head_size,bias=False)
    self.value = nn.Linear(n_embedd,head_size,bias=False)

  def forward(self,x):
    q = self.querry(x)
    k = self.key(x)
    v = self.value(x)
    weights = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
    weights = weights.masked_fill(torch.tril(torch.ones(weights.shape))==0,float('-inf'))
    weights = torch.softmax(weights,dim=-1)
    out = weights @ v
    return out



In [17]:
class LayerNorm(nn.Module):
  def __init__(self,n_embedd):
    super().__init__()
    self.ln1 = nn.LayerNorm(n_embedd)
  def forward(self,x,contextual_embedding):
    y = x + contextual_embedding
    return self.ln1(y)

In [18]:
class multihead_masked_self_attention(nn.Module):
    def __init__(self, no_of_heads, n_embedd):
        super().__init__()
        self.head_size = n_embedd // no_of_heads
        self.heads = nn.ModuleList([
            Masked_self_attention(n_embedd,self.head_size)
            for _ in range(no_of_heads)
        ])
        self.projection = nn.Linear(n_embedd, n_embedd)
    def forward(self, embedding):
        out = torch.cat(
            [head(embedding) for head in self.heads],
            dim=-1
        )
        return self.projection(out)


In [19]:
class feed_forward(nn.Module):
  def __init__(self,n_embedd):
    super().__init__()
    self.seq = nn.Sequential(
        nn.Linear(n_embedd,4*n_embedd),
        nn.ReLU(),
        nn.Linear(4*n_embedd,n_embedd)
    )
  def forward(self,x):
    return self.seq(x)

In [20]:
class Block(nn.Module):
  def __init__(self,n_embedd):
    super().__init__()
    self.msa = multihead_masked_self_attention(8,n_embedd)
    self.layernorm1 = LayerNorm(n_embedd)
    self.ffwd  = feed_forward(n_embedd)
    self.layernorm2 = LayerNorm(n_embedd)
  def forward(self,x):
    contextual_embedding = self.msa(x)
    norm1_embedding = self.layernorm1(x,contextual_embedding)
    ffwd_embedding = self.ffwd(norm1_embedding)
    norm2_embedding = self.layernorm2(norm1_embedding,ffwd_embedding)
    return norm2_embedding



In [21]:
class language_head(nn.Module):
  def __init__(self,n_embedd,vocab_size):
    super().__init__()
    self.seq = nn.Sequential(
        nn.Linear(n_embedd,vocab_size)
    )
  def forward(self,x):
    return self.seq(x)

In [22]:
torch.manual_seed(42)

class gpt(nn.Module):
  def __init__(self,n_embedd,vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,n_embedd)
    self.posi_embedding = nn.Embedding(block_size,n_embedd)
    self.transformer_block = nn.ModuleList(Block(n_embedd) for _ in range(4))
    self.lang_head = language_head(n_embedd,vocab_size)

  def forward(self,x,target=None):
      batch_index,token_size = x.shape
      token_embedd = self.embedding(x)
      posi_embedd = self.posi_embedding(torch.arange(token_size))
      x = token_embedd + posi_embedd
      for block in self.transformer_block:
        x = block(x)
      logits = self.lang_head(x)
      loss = None
      if target is not None:
        B,T,C = logits.shape
        logits = logits.view(B*T,C)
        target = target.view(B*T)
        loss = nn.functional.cross_entropy(logits,target)
      return logits,loss


  @torch.no_grad()
  def genrate(self,input,max_new_tokens):
      for _ in range(max_new_tokens):
        input_cond = input[:, -block_size:]
        logits, loss = self(input_cond)
        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input = torch.cat((input, next_token), dim=1)
      return input



In [23]:
model = gpt(n_embedd,vocab_size)
optimizer = torch.optim.AdamW(model.parameters(),
                              lr=0.0001)

[decoder archi](https://waylandz.com/blog/transformer-architecture/)


In [24]:
block_size = 64
model = gpt(n_embedd, vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)

In [32]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        loss_list = []
        for _ in range(100):
            xb, yb = get_batch(split)
            logits, loss = model(xb, yb)
            loss_list.append(loss.item())
        losses[split] = sum(loss_list) / len(loss_list)
    model.train()
    return losses


max_iters = 5000
eval_interval = 500
for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"Step {iter}: Train Loss: {losses['train']:.4f}, Val Loss: {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
# Final evaluation
losses = estimate_loss()
print("Final:", losses)

Step 0: Train Loss: 1.8073, Val Loss: 1.9250
Step 500: Train Loss: 1.7562, Val Loss: 1.8825
Step 1000: Train Loss: 1.7462, Val Loss: 1.8816
Step 1500: Train Loss: 1.7093, Val Loss: 1.8517
Step 2000: Train Loss: 1.7063, Val Loss: 1.8384
Step 2500: Train Loss: 1.6682, Val Loss: 1.8244
Step 3000: Train Loss: 1.6758, Val Loss: 1.8041
Step 3500: Train Loss: 1.6200, Val Loss: 1.7951
Step 4000: Train Loss: 1.5975, Val Loss: 1.7912
Step 4500: Train Loss: 1.6108, Val Loss: 1.7704
Final: {'train': 1.6064327812194825, 'val': 1.8090178430080415}


In [33]:
model.eval()
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.genrate(context, 500)
print(decode(generated[0].tolist()))


And this dad of dend his fath,
I somethed we head, not thy goness lianl the cound man
To baids death to my chame as curse wit.

ire
Unspearting other all of Come,
I'll tatchtard me time stand senoth betrees,
And daon of the hoCle.

PPETES:
Madiets to uther.

DUKE OF YURENCE:
Lay dismoth tadile: have paice the I dy reshold
in hat siul stal
And will grace, this his burne ones, thou with though
thopsed me grave thy words: those risonat'st chim
a hear a not scitter: go he oursh will.

BUCKINGHNGHENV


In [34]:
prompt = "ROMEO:"
context = torch.tensor([encode(prompt)], dtype=torch.long)
generated = model.genrate(context, 500)
print(decode(generated[0].tolist()))

ROMEO:
Brecate hand impersole rewards!

BRUTUCKINGHAM:
How Lady good gieft do us lloog to Gufess:
Indoul foll, suchile enswer thus ochas poson:
Be speak you histandons Contifue anmurgian the scollows
His dangth take his greesolied from apied you of peace?

Shing Cail is.
What, no provesceds cond before his duked love
Would not dear eime maine: that shall peace live
Untion, to be widness agest thd grovess this?

ANGLONTIU:
As if merch the served ento the bagnd,: I will which whom
Will a grace be out ar


In [28]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'block_size': block_size,
    'n_embedd': n_embedd,
    'vocab_size': vocab_size,
    'chars': chars
}, '/content/drive/MyDrive/gpt_block64.pth')

# Final: {'train': 1.6064327812194825, 'val': 1.8090178430080415}

In [30]:
# torch.save(model.state_dict(),'gpt_block8.pth')